# Your First Transformer Models With pipeline

**What you'll learn**
- Run ready-made models from the Hugging Face Hub with `pipeline`
- Sentiment and emotion on support tickets, and why their labels are not ours
- Route tickets with zero-shot classification, using our own four labels
- Ask yes or no questions, each label scored on its own
- Ask a chat model to pick the team in words
- Score every model on all 40 tickets against the gold answers

## Setup
- The 40 support tickets used across the course
- Each has a gold `department`, `urgency` and `refund`
- Library warnings are switched off to keep the outputs readable

In [ ]:
import pandas as pd
from transformers import pipeline, logging

logging.set_verbosity_error()

In [ ]:
tickets = pd.read_csv("data/tickets.csv")
tickets.head()

In [ ]:
tickets["department"].value_counts()

In [ ]:
texts = tickets["text"].tolist()  # what the models read
gold = tickets["department"].tolist()  # the right answers, kept apart
len(texts)

## Sentiment Analysis on Support Tickets
- `text-classification` with a model fine-tuned on movie reviews (SST-2)
- It knows two labels only: `POSITIVE` and `NEGATIVE`

In [ ]:
texts[0]

In [ ]:
sentiment = pipeline("text-classification",
                     model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
                     device_map="cuda")

response = sentiment(texts[0])
response

- The sales request: a question about two price plans, not a complaint

In [ ]:
texts[3]

In [ ]:
sentiment(texts[3])

### Sentiment on All 40 Tickets
- A list in, one answer per ticket out, in the same order

In [ ]:
results = sentiment(texts)
results[:3]

In [ ]:
labels = []
for r in results:
    labels.append(r["label"])

tickets["sentiment"] = labels
pd.crosstab(tickets["department"], tickets["sentiment"])

## Emotion Classification With 28 Labels
- Same task, a model trained on 28 emotions (GoEmotions)
- `top_k=3`: the three highest scores

In [ ]:
emotions = pipeline("text-classification",
                    model="SamLowe/roberta-base-go_emotions",
                    top_k=3,
                    device_map="cuda")

emotions(texts[0])

- The sarcastic ticket: it says great and love, and means neither

In [ ]:
texts[10]

In [ ]:
sentiment(texts[10])

In [ ]:
emotions(texts[10])

### Emotions on All 40 Tickets
- The top emotion for each ticket

In [ ]:
results = emotions(texts)

top = []
for r in results:
    top.append(r[0]["label"])  # r is the top three, best first

tickets["emotion"] = top
tickets["emotion"].value_counts()

## Routing With Zero-Shot Classification
- Any labels you like, no training
- An NLI model scores "This support ticket is about {label}." for each label

In [ ]:
zero_shot = pipeline("zero-shot-classification",
                     model="facebook/bart-large-mnli",
                     device_map="cuda")
DEPTS = ["billing", "technical", "account", "sales"]

response = zero_shot(texts[0],
                     candidate_labels=DEPTS,
                     hypothesis_template="This support ticket is about {}.")

response

In [ ]:
gold[0]

### Zero-Shot on All 40 Tickets
- The labels come back sorted, so the first one is the model's pick

In [ ]:
results = zero_shot(texts,
                    candidate_labels=DEPTS,
                    hypothesis_template="This support ticket is about {}.")

picks = []
top_scores = []
for r in results:
    picks.append(r["labels"][0])
    top_scores.append(r["scores"][0])

tickets["zero_shot"] = picks
tickets[["text", "department", "zero_shot"]].head()

In [ ]:
correct = tickets["zero_shot"] == tickets["department"]
correct.mean()

In [ ]:
pd.crosstab(tickets["department"], tickets["zero_shot"])

- Rows are the gold department, columns are the model's pick
- Compare the accuracy with how sure it was on average

In [ ]:
sum(top_scores) / len(top_scores)

### Yes or No Questions With Zero-Shot Classification
- `multi_label=True`: each label scored on its own, the scores need not sum to 1
- The refund question has a gold answer in the `refund` column

In [ ]:
LABELS = ["asks for money back", "asks for technical support"]

response = zero_shot(texts[0:4],
                     candidate_labels=LABELS,
                     hypothesis_template="This support ticket {}.",
                     multi_label=True)
response

### The Refund Question on All 40 Tickets
- A score above 0.5 counts as a yes

In [ ]:
results = zero_shot(texts,
                    candidate_labels=["asks for money back"],
                    hypothesis_template="This support ticket {}.",
                    multi_label=True)

scores = []
for r in results:
    scores.append(r["scores"][0])

tickets["refund_score"] = scores
tickets["refund_pred"] = tickets["refund_score"] > 0.5
tickets[["text", "refund", "refund_score", "refund_pred"]].head()

In [ ]:
(tickets["refund_pred"] == tickets["refund"]).mean()

In [ ]:
pd.crosstab(tickets["refund"], tickets["refund_pred"])

## Ask a Chat Model to Classify the Ticket in Words
- `text-generation` with Qwen2.5-0.5B-Instruct
- The answer comes back as words, with no score

In [ ]:
chat = pipeline("text-generation",
                model="Qwen/Qwen2.5-0.5B-Instruct",
                device_map="cuda")

ticket = texts[2]
question = (f"Ticket: {ticket}\n"
            "Which team should handle it: billing, technical, account or sales? "
            "Answer in one word.")

msgs = [{"role": "user", "content": question}]
out = chat(msgs, max_new_tokens=20, do_sample=False)
out

In [ ]:
out[0]["generated_text"][-1]["content"]  # the last message is the model's reply

In [ ]:
gold[2]

### The Chat Model on All 40 Tickets
- Same question for every ticket, the reply tidied to one lowercase word

In [ ]:
answers = []
for ticket in texts:
    question = (f"Ticket: {ticket}\n"
                "Which team should handle it: billing, technical, account or sales? "
                "Answer in one word.")
    msgs = [{"role": "user", "content": question}]
    out = chat(msgs, max_new_tokens=20, do_sample=False)
    reply = out[0]["generated_text"][-1]["content"]
    answers.append(reply.strip().strip(".").lower())

tickets["chat"] = answers
tickets["chat"].value_counts()

In [ ]:
(tickets["chat"] == tickets["department"]).mean()

In [ ]:
pd.crosstab(tickets["department"], tickets["chat"])

## Every Model on the Same 40 Tickets
- Department accuracy for the two models that can name a department
- Always answering the biggest department is the bar to beat

In [ ]:
scoreboard = {
    "always 'technical'": (tickets["department"] == "technical").mean(),
    "zero-shot, our four labels": (tickets["zero_shot"] == tickets["department"]).mean(),
    "chat model, in words": (tickets["chat"] == tickets["department"]).mean(),
}
pd.Series(scoreboard)